# SentinelSat
 - [GitHub](https://github.com/sentinelsat/sentinelsat)
 - [ReadTheDocs](https://sentinelsat.readthedocs.io/en/stable/api_overview.html#lta-products)
 

### Load

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# %load import.py
# 
import numpy as np
import pandas as pd

#
from sentinelsat import SentinelAPI, read_geojson, geojson_to_wkt
from datetime import date

#
import rioxarray
import geopandas as gpd
import rasterio as rio

#
from matplotlib import pyplot
from rasterio.plot import show

#
from sqlalchemy import create_engine # query PostGIS
from sqlalchemy import inspect

#
import osmnx as ox

#
from shapely.geometry import Polygon, box
import shapely.ops as so

#
import json

#
import os
from pathlib import Path
import fnmatch
import glob

#
from osgeo import gdal

#
import random

#
import shutil

#
from tqdm import tqdm
import time

## Study Area

In [ ]:
# sa_fil = "italy_center_south.geojson"
sa_fil = "naples_metropolytan.geojson"

In [ ]:
sa_j = read_geojson(sa_fil)

In [ ]:
type(sa_j)

In [ ]:
sa = geopandas.read_file(sa_fil)

In [ ]:
type(sa)

In [ ]:
sa.plot()

### Geospatial raster and Vector data with Python

 - [EGU short course 2023](https://github.com/esciencecenter-digital-skills/2023-04-25-ds-geospatial-python-EGU)

### Check out `RapidEye time series for Sentinel-2`

 - [ESA link](https://earth.esa.int/eogateway/catalog/rapideye-time-series-for-sentinel-2)

## PROCEDURE

### Credentials / API

In [ ]:
sentinelsat_credentials_file = "sentinelsat_credentials.json"

In [ ]:
# Opening JSON file
with open( sentinelsat_credentials_file ) as json_file:
    sensat = json.load(json_file)
 
    # Print the type of data variable
#    print("Type:", type(data))
 
    # Print the data of dictionary
#    print("\nUser      :", data['user'])
#    print("\nPassword  :", data['password'])

In [ ]:
user = sensat['user']
pswd = sensat['password']
sentinelSat_endpoint = "https://apihub.copernicus.eu/apihub"

In [ ]:
api = SentinelAPI(user, pswd, sentinelSat_endpoint)

In [ ]:
api.session

### Request Sat Products

In [ ]:
date_FROM = date(2023,4,1)
date_TO   = date(2023,5,1)

In [ ]:
platform_name = "Sentinel-2"

In [ ]:
cloud_coverage = (0,10)

In [ ]:
# search by polygon, time, and SciHub query keywords
products = api.query( geojson_to_wkt(sa_j),
                      date = ( date_FROM, date_TO ),
                      platformname = platform_name,
                      cloudcoverpercentage = cloud_coverage
                    )

In [ ]:
len(products)

### GeoDataFrame from Products

In [ ]:
gdf = api.to_geodataframe(products)

In [ ]:
type(gdf)

In [ ]:
gdf.head(1)

In [ ]:
gdf.plot()

In [ ]:
gdf.bounds

### Select Products

Get the list of first 5 keys:

In [ ]:
list(products.keys())[0:5]

Select only the first product:

In [ ]:
sel_products = dict(list(products.items())[:1])

In [ ]:
api.get_product_odata( list(sel_products.keys())[0] )

In [ ]:
api.to_geodataframe(sel_products)

#### Save GeoDF

In [ ]:
sel_prod_gdf = api.to_geodataframe(sel_products)

In [ ]:
sel_prod_gdf.to_file("sel_products.geojson",driver="GeoJSON")

### Download Selected Products

In [ ]:
# Activate only if required:
# api.download_all(sel_products,directory_path="data/")

### Map Selected Products

In [ ]:
ras_path_20m = "data/S2A_MSIL1C_20230427T095031_N0509_R079_T33TVF_20230427T115004.SAFE/GRANULE/L1C_T33TVF_A040974_20230427T095813/IMG_DATA/"

In [ ]:
raster = rioxarray.open_rasterio(ras_path_20m + "T33TVF_20230427T095031_TCI.jp2")

In [ ]:
raster

In [ ]:
raster.rio.crs

In [ ]:
raster.plot()

In [ ]:
raster.plot.imshow()

### Get Sentinel-2 tiling grid

In [ ]:
!wget https://sentinel.esa.int/documents/247904/1955685/S2A_OPER_GIP_TILPAR_MPC__20151209T095117_V20150622T000000_21000101T000000_B00.kml

In [ ]:
s2_tile_grid = geopandas.read_file("S2A_OPER_GIP_TILPAR_MPC__20151209T095117_V20150622T000000_21000101T000000_B00.kml")

In [ ]:
s2_tile_grid.head(20).plot()